# Week 3 — The Forward Diffusion Process
### Diffusion Models from Scratch — SoC 2026

This is the most math-heavy week. We're building a `NoiseScheduler` that implements the forward diffusion process — given a clean image and a timestep `t`, produce the correctly-noised version in **one closed-form step**.

**Key insight this week:** We don't need to iterate `t` times to get `x_t`. The magic of the reparameterization trick lets us jump directly to any timestep.

**Setup:** `Runtime → Change runtime type → T4 GPU → Save`

## Section 0 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import math

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Section 1 — Math Review Before Coding

Before touching a keyboard, let's lock in the notation.

The forward process is a Markov chain that gradually adds Gaussian noise:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} \, x_{t-1}, \, \beta_t I)$$

where $\beta_t \in (0, 1)$ is the **noise schedule** — how much noise to add at step $t$.

Define:
- $\alpha_t = 1 - \beta_t$  
- $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ (cumulative product)

Then the **closed-form** for sampling $x_t$ directly from $x_0$ is:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} \, x_0, \, (1 - \bar{\alpha}_t) I)$$

Using the reparameterization trick:

$$x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This is the one equation you need to implement.

### ❓ Conceptual Question 1
**Why can we sample `x_t` directly from `x_0` in one step instead of iterating `t` times? What's the math behind that, and why does it matter for training?**

**Your answer:**

The key is that a sum of independent Gaussians is itself Gaussian. Each step in the forward process adds Gaussian noise, and if you chain together `t` steps of Gaussian noise additions, the result is still a Gaussian — just with a different mean and variance. You can work out the math by repeatedly applying the transition equation, and it simplifies cleanly because of how Gaussian variance addition works.

Specifically: at step 1, $x_1 = \sqrt{\alpha_1} x_0 + \sqrt{1-\alpha_1}\epsilon_1$. At step 2, substitute $x_1$ in: $x_2 = \sqrt{\alpha_2}(\sqrt{\alpha_1} x_0 + \sqrt{1-\alpha_1}\epsilon_1) + \sqrt{1-\alpha_2}\epsilon_2 = \sqrt{\alpha_1\alpha_2}x_0 + \text{noise}$. The noise terms — both Gaussians — combine into a single Gaussian, and you get $\bar{\alpha}_t = \prod \alpha_s$ as the cumulative product of all the scaling factors.

**Why this matters for training:** If you had to iterate `t` times to get `x_t` from `x_0`, training would be astronomically slow. With T=1000 timesteps and millions of training samples, you'd need 1000 forward passes through the noise chain per sample per gradient update. With the closed-form, you can sample any timestep `t` uniformly at random and compute the correctly-noised `x_t` in a single step — O(1) instead of O(T). This makes the training loop practical.

### ❓ Conceptual Question 2
**What's the difference between β, α, and ᾱ (alpha bar)?**

**Your answer:**

- **β_t** (beta): The noise schedule itself — how much variance to inject at timestep `t`. It's a small positive number, typically ranging from something like 0.0001 to 0.02. Higher β means more noise added in that one step.

- **α_t = 1 - β_t** (alpha): The signal retention factor. Since we're adding noise with variance β_t, we need to scale the signal by √α_t to keep the total variance from exploding. It's always slightly less than 1.

- **ᾱ_t = ∏_{s=1}^{t} α_s** (alpha bar): The cumulative product of all α's up to timestep `t`. This is the most important one for the closed-form sampling — ᾱ_t tells you exactly how much of the original signal survives after `t` steps of noising. √ᾱ_t is the coefficient on x_0 in the closed-form, and √(1-ᾱ_t) is the coefficient on the noise. At t=0, ᾱ_0 = 1 (pure signal). At t=T, ᾱ_T ≈ 0 (pure noise).

### ❓ Conceptual Question 3
**Why do we need a noise schedule at all — why not just add full noise at once?**

**Your answer:**

The noise schedule gives the model a **curriculum**: it has to learn to denoise at every level of noisiness, from slightly-noisy (easy) to almost-pure-noise (hard). If you just added full noise all at once, there'd be nothing for the model to gradually learn — it'd be trying to reconstruct the image from what's essentially random noise with no useful signal left.

The gradual schedule is also what makes the reverse process tractable. The model needs to learn $p_\theta(x_{t-1} | x_t)$ — how to take a small denoising step. If each step is small, the conditional distribution $q(x_{t-1} | x_t, x_0)$ is also approximately Gaussian, which is what lets us parameterize the reverse process as a Gaussian and train it with a simple MSE loss. If the forward steps were huge, the reverse distribution would be complicated and non-Gaussian, and we couldn't fit it with a simple parametric model.

Essentially: small steps forward → tractable reverse steps → trainable model.

### ❓ Conceptual Question 4
**What does the reparameterization trick give us?**

**Your answer:**

The reparameterization trick separates the **stochasticity** from the **computation**. Instead of sampling $x_t \sim \mathcal{N}(\sqrt{\bar\alpha_t}x_0, (1-\bar\alpha_t)I)$ directly (which you can't backpropagate through), you write it as $x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\cdot\epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$.

Now the randomness lives entirely in $\epsilon$, which is sampled independently with no parameters attached to it. The parameters enter through the deterministic scaling $\sqrt{\bar\alpha_t}$ and $\sqrt{1-\bar\alpha_t}$. This means gradients can flow through the sampling operation, which is essential for training.

For the diffusion model specifically: during training, we sample $\epsilon$, compute $x_t$, predict $\hat\epsilon$ with the model, and compute the loss $\|\epsilon - \hat\epsilon\|^2$. The reparameterization is what lets us write this clean noise-prediction objective — without it, we couldn't frame training as "predict the noise we added".

## Section 2 — NoiseScheduler Class

This is the core deliverable. We implement both **linear** and **cosine** schedules.

In [ ]:
class NoiseScheduler:
    """
    DDPM noise scheduler supporting linear and cosine beta schedules.

    After init, the following tensors are available on `device`:
      betas              : (T,)  — noise variance at each step
      alphas             : (T,)  — 1 - betas
      alphas_cumprod     : (T,)  — ᾱ_t = ∏_{s=1}^{t} α_s
      sqrt_alphas_cumprod        : (T,)
      sqrt_one_minus_alphas_cumprod : (T,)
    """

    def __init__(
        self,
        num_timesteps: int = 1000,
        schedule: str = "linear",
        beta_start: float = 1e-4,
        beta_end: float = 0.02,
        device=None,
    ):
        self.T = num_timesteps
        self.device = device or torch.device("cpu")

        betas = self._make_betas(schedule, beta_start, beta_end)

        # Precompute everything we'll need at training and sampling time
        self.betas = betas.to(self.device)
        self.alphas = (1.0 - betas).to(self.device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)  # ᾱ_t

        self.sqrt_alphas_cumprod = self.alphas_cumprod.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - self.alphas_cumprod).sqrt()

        # For the reverse process (Week 4 will use these)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )

    def _make_betas(self, schedule: str, beta_start: float, beta_end: float) -> torch.Tensor:
        if schedule == "linear":
            # Linearly spaced betas from beta_start to beta_end
            return torch.linspace(beta_start, beta_end, self.T)

        elif schedule == "cosine":
            # Cosine schedule from Nichol & Dhariwal (Improved DDPM, 2021)
            # ᾱ_t = cos^2(((t/T + s) / (1 + s)) * π/2)  where s=0.008
            s = 0.008
            steps = torch.arange(self.T + 1, dtype=torch.float64)
            f = torch.cos(((steps / self.T + s) / (1 + s)) * math.pi / 2) ** 2
            alphas_cumprod = f / f[0]  # normalize so ᾱ_0 = 1
            betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
            # Clip to avoid numerical issues
            return torch.clamp(betas, min=0.0, max=0.999).float()

        else:
            raise ValueError(f"Unknown schedule: {schedule!r}. Choose 'linear' or 'cosine'.")

    def add_noise(
        self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Forward process: compute x_t from x_0 in one step.

        x_t = sqrt(ᾱ_t) * x_0 + sqrt(1 - ᾱ_t) * ε

        Args:
            x0  : (B, C, H, W)  clean images, normalized to [-1, 1]
            t   : (B,)          integer timestep indices in [0, T-1]
            noise: (B, C, H, W) optional; sampled if not provided

        Returns:
            xt    : noised images
            noise : the noise that was added (needed for training)
        """
        if noise is None:
            noise = torch.randn_like(x0)

        # Gather the right coefficients for each item in the batch
        # t is (B,), we need (B, 1, 1, 1) to broadcast over (B, C, H, W)
        sqrt_alpha_bar = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)

        xt = sqrt_alpha_bar * x0 + sqrt_one_minus * noise
        return xt, noise

    def snr(self) -> torch.Tensor:
        """Signal-to-noise ratio: ᾱ_t / (1 - ᾱ_t) for all t."""
        return self.alphas_cumprod / (1.0 - self.alphas_cumprod)


# Instantiate both schedules
linear_sched = NoiseScheduler(num_timesteps=1000, schedule="linear", device=device)
cosine_sched = NoiseScheduler(num_timesteps=1000, schedule="cosine", device=device)

print(f"Linear  | β range: [{linear_sched.betas.min():.6f}, {linear_sched.betas.max():.6f}]")
print(f"Cosine  | β range: [{cosine_sched.betas.min():.6f}, {cosine_sched.betas.max():.6f}]")
print(f"ᾱ at t=0 (linear): {linear_sched.alphas_cumprod[0].item():.6f}")
print(f"ᾱ at t=999 (linear): {linear_sched.alphas_cumprod[-1].item():.6f}")

## Section 3 — Unit Tests

Before we trust the scheduler, let's verify the math.

In [ ]:
def test_pure_noise_at_T(scheduler, tol=0.1):
    """
    At t=T-1 (the final timestep), x_T should be ≈ N(0, I).
    We test by sampling many x_T values and checking mean≈0, std≈1.
    """
    N = 2000
    x0 = torch.zeros(N, 1, 28, 28, device=device)  # start from all-zeros
    t = torch.full((N,), scheduler.T - 1, device=device, dtype=torch.long)
    xt, _ = scheduler.add_noise(x0, t)
    mean = xt.mean().item()
    std  = xt.std().item()
    assert abs(mean) < tol, f"Mean at T should be ≈0, got {mean:.4f}"
    assert abs(std - 1.0) < tol, f"Std at T should be ≈1, got {std:.4f}"
    print(f"  [{scheduler.__class__.__name__}] x_T mean={mean:.4f}, std={std:.4f}  ✓")


def test_identity_at_t0(scheduler, tol=1e-5):
    """
    At t=0, x_t should be essentially x_0 (very little noise added).
    sqrt(ᾱ_0) should be very close to 1, sqrt(1-ᾱ_0) close to 0.
    """
    x0 = torch.randn(4, 1, 28, 28, device=device)
    t  = torch.zeros(4, device=device, dtype=torch.long)
    zero_noise = torch.zeros_like(x0)
    xt, _ = scheduler.add_noise(x0, t, noise=zero_noise)
    # With zero noise: xt = sqrt(ᾱ_0) * x0
    # sqrt(ᾱ_0) should be very close to 1 for both schedules
    ratio = (xt / (x0 + 1e-9)).mean().item()
    print(f"  xt/x0 at t=0 ≈ {ratio:.6f}  (expect ≈ {scheduler.sqrt_alphas_cumprod[0].item():.6f})")


def test_noise_increases_with_t(scheduler):
    """
    The fraction of signal should strictly decrease as t increases.
    """
    abar = scheduler.alphas_cumprod.cpu()
    assert (abar[1:] < abar[:-1]).all(), "ᾱ_t should be strictly decreasing"
    print(f"  ᾱ strictly decreasing  ✓")


def test_output_shape(scheduler):
    x0 = torch.randn(8, 1, 28, 28, device=device)
    t  = torch.randint(0, scheduler.T, (8,), device=device)
    xt, noise = scheduler.add_noise(x0, t)
    assert xt.shape == x0.shape, f"Shape mismatch: {xt.shape}"
    assert noise.shape == x0.shape
    print(f"  Output shape {xt.shape}  ✓")


print("=== Linear Schedule Tests ===")
test_pure_noise_at_T(linear_sched)
test_identity_at_t0(linear_sched)
test_noise_increases_with_t(linear_sched)
test_output_shape(linear_sched)

print("\n=== Cosine Schedule Tests ===")
test_pure_noise_at_T(cosine_sched)
test_identity_at_t0(cosine_sched)
test_noise_increases_with_t(cosine_sched)
test_output_shape(cosine_sched)

print("\nAll tests passed!")

## Section 4 — Visualize the Noising Trajectory

Pick one MNIST image and show it at t = 0, 100, 250, 500, 750, 999.

In [ ]:
# Load one clean MNIST image, normalize to [-1, 1]
mnist = datasets.MNIST(
    root="./data", train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),  # [0,1] -> [-1, 1]
    ])
)
x0, label = mnist[7]  # pick the digit '7'
x0 = x0.unsqueeze(0).to(device)  # (1, 1, 28, 28)

timesteps_to_show = [0, 100, 250, 500, 750, 999]

def show_noising_trajectory(scheduler, title):
    fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(14, 2.5))
    torch.manual_seed(0)  # fixed noise seed for reproducibility
    for ax, t_val in zip(axes, timesteps_to_show):
        t = torch.tensor([t_val], device=device, dtype=torch.long)
        torch.manual_seed(0)  # same noise at each t so we isolate schedule effect
        xt, _ = scheduler.add_noise(x0, t)
        # Denormalize: [-1,1] -> [0,1]
        img = (xt.squeeze().cpu().clamp(-1, 1) + 1) / 2
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"t={t_val}", fontsize=10)
        ax.axis("off")
    fig.suptitle(title, fontsize=12, y=1.05)
    plt.tight_layout()
    plt.show()

show_noising_trajectory(linear_sched, "Linear Schedule — Noising Trajectory")
show_noising_trajectory(cosine_sched, "Cosine Schedule — Noising Trajectory")

## Section 5 — SNR Curves: Linear vs Cosine

The **signal-to-noise ratio** (SNR) is $\bar\alpha_t / (1 - \bar\alpha_t)$. It tells you how much useful signal remains at each timestep. We want to see how quickly information is destroyed under each schedule.

In [ ]:
timesteps = torch.arange(1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Plot 1: Beta schedules ---
axes[0].plot(linear_sched.betas.cpu(), label="linear", color="steelblue")
axes[0].plot(cosine_sched.betas.cpu(), label="cosine", color="tomato")
axes[0].set_title("β_t (noise schedule)"); axes[0].set_xlabel("timestep t")
axes[0].set_ylabel("β_t"); axes[0].legend(); axes[0].grid(alpha=0.3)

# --- Plot 2: Alpha bar (signal retention) ---
axes[1].plot(linear_sched.alphas_cumprod.cpu(), label="linear", color="steelblue")
axes[1].plot(cosine_sched.alphas_cumprod.cpu(), label="cosine", color="tomato")
axes[1].set_title("ᾱ_t (signal retention)"); axes[1].set_xlabel("timestep t")
axes[1].set_ylabel("ᾱ_t"); axes[1].legend(); axes[1].grid(alpha=0.3)

# --- Plot 3: SNR ---
lin_snr = linear_sched.snr().cpu()
cos_snr = cosine_sched.snr().cpu()
axes[2].semilogy(lin_snr, label="linear", color="steelblue")
axes[2].semilogy(cos_snr, label="cosine", color="tomato")
axes[2].set_title("SNR = ᾱ_t / (1 - ᾱ_t) [log scale]")
axes[2].set_xlabel("timestep t"); axes[2].set_ylabel("SNR")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Linear  | ᾱ at t=500:  {linear_sched.alphas_cumprod[500].item():.6f}")
print(f"Cosine  | ᾱ at t=500:  {cosine_sched.alphas_cumprod[500].item():.6f}")
print(f"Linear  | ᾱ at t=999:  {linear_sched.alphas_cumprod[-1].item():.6f}")
print(f"Cosine  | ᾱ at t=999:  {cosine_sched.alphas_cumprod[-1].item():.6f}")

### ❓ Conceptual Question 5
**Why does the cosine schedule outperform the linear schedule empirically?**

**Your answer:**

Look at the ᾱ_t plot — the linear schedule destroys information very quickly in the early-to-middle timesteps. By around t=500 the image is already nearly unrecognizable under the linear schedule (ᾱ ≈ 0.01 or lower), meaning the model has to do most of its learning in that narrow range of low timesteps and the high-noise timesteps are basically learning nothing useful. The signal is gone too fast.

The cosine schedule has a much more gradual, S-shaped decay. It spends more time in the "intermediate" noise levels where there's still enough signal to learn something meaningful, but also enough noise to be challenging. Think of it as a better curriculum — more of the timesteps are "useful" training signal rather than being either trivially easy (very low t) or completely hopeless (very high t under linear).

There's also a practical stability benefit: the linear schedule can produce very small ᾱ_T values that lead to numerical issues when computing sqrt(ᾱ_t) near t=T. The cosine schedule clips the minimum beta and handles this more gracefully. Nichol & Dhariwal's paper specifically showed this leads to better sample quality and a smoother loss landscape, particularly on low-resolution images where the linear schedule wastes too many training steps on near-pure noise.

### ❓ Conceptual Question 6
**Why must image data be normalized to [-1, 1] before adding noise?**

**Your answer:**

The closed-form update `x_t = sqrt(ᾱ_t) * x_0 + sqrt(1 - ᾱ_t) * ε` mixes the original image with standard Gaussian noise `ε ~ N(0, I)`, which has mean 0 and unit variance by construction. For that mixture to behave the way the schedule's math assumes — and for `x_T` to actually converge to something close to `N(0, I)` as `ᾱ_T → 0` — the *signal* term `x_0` needs to be on the same scale as the noise it's being mixed with, i.e. roughly zero-mean and unit-scale.

Raw image data in `[0, 1]` (or `[0, 255]`) is not zero-mean — it's a non-negative range with a mean well above 0. If I added unit Gaussian noise directly to `[0, 1]`-range pixels, the schedule's coefficients (calibrated assuming the data and noise are on comparable scales) would be wrong: at low `t`, the "noise" would dominate a signal that's already small relative to it, and at high `t`, `x_T` would settle around the data's mean instead of `N(0, I)`. Normalizing to `[-1, 1]` centers the data around 0, which is what makes the forward process's noise schedule and the unit-tests checking `x_T ≈ N(0, I)` actually valid.

### ❓ Conceptual Question 7
**As t → T, what does x_t approach? Why does that matter for sampling?**

**Your answer:**

As `t → T`, `ᾱ_t → 0` (verified above — both schedules' `alphas_cumprod` trail off toward a tiny value by `t=999`), so the closed-form `x_t = sqrt(ᾱ_t) * x_0 + sqrt(1 - ᾱ_t) * ε` collapses to just `x_t ≈ ε`, i.e. pure standard Gaussian noise with essentially no trace of `x_0` left. That's exactly what the unit tests in Section 3 (`test_pure_noise_at_T`) and the pixel-histogram check in Section 7 verify directly — mean ≈ 0, std ≈ 1, matching `N(0, I)`.

This matters enormously for sampling (Week 4): the *entire reverse process* depends on `x_T` being indistinguishable from pure noise, because that's the only thing I'll actually have at generation time — there's no real image to start from. If `x_T` retained some residual signal from `x_0` instead of being genuinely `N(0, I)`, then sampling by starting from `torch.randn(...)` wouldn't match the distribution the model was actually trained to reverse, and the reverse process would be denoising from the wrong starting distribution — the generated samples would inherit whatever bias was left over, rather than being able to generate genuinely novel images from scratch.

## Section 6 — Dataset Pipeline for Training

The final piece: a Dataset class that produces `(x_0, x_t, t, noise)` tuples — exactly what the UNet in Week 4 will consume.

In [ ]:
class DiffusionDataset(torch.utils.data.Dataset):
    """
    Wraps a clean image dataset and produces diffusion training tuples.
    Each call to __getitem__ samples a random timestep and returns:
      x0    : clean image, normalized to [-1, 1]
      xt    : noised image at timestep t
      t     : integer timestep (scalar)
      noise : the Gaussian noise added (the training target)
    """
    def __init__(self, scheduler: NoiseScheduler, train: bool = True):
        self.scheduler = scheduler
        self.mnist = datasets.MNIST(
            root="./data", train=train, download=True,
            transform=transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,)),  # [-1, 1]
            ])
        )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        x0, _ = self.mnist[idx]                          # (1, 28, 28)
        t = torch.randint(0, self.scheduler.T, (1,)).long()  # random timestep
        x0_dev = x0.unsqueeze(0).to(self.scheduler.device)   # add batch dim
        xt, noise = self.scheduler.add_noise(x0_dev, t)
        return x0.squeeze(0), xt.squeeze(0), t.squeeze(0), noise.squeeze(0)


# Test the dataset
diff_ds = DiffusionDataset(linear_sched)
x0_s, xt_s, t_s, noise_s = diff_ds[0]
print(f"x0 shape:    {x0_s.shape}")
print(f"xt shape:    {xt_s.shape}")
print(f"t:           {t_s.item()}")
print(f"noise shape: {noise_s.shape}")
print(f"x0 range:    [{x0_s.min():.2f}, {x0_s.max():.2f}]  (should be in [-1, 1])")

# Verify the equation: xt = sqrt(abar) * x0 + sqrt(1-abar) * noise
t_idx = t_s.long()
sqrt_abar = linear_sched.sqrt_alphas_cumprod[t_idx].item()
sqrt_1m   = linear_sched.sqrt_one_minus_alphas_cumprod[t_idx].item()
xt_check = sqrt_abar * x0_s.cpu() + sqrt_1m * noise_s.cpu()
err = (xt_check - xt_s.cpu()).abs().max().item()
print(f"\nClosed-form check: max|xt_reconstructed - xt| = {err:.2e}  (should be ~0)")

## Section 7 — Visualize a Full Noising Distribution

Show that at t=T, the output really is drawn from N(0, I).

In [ ]:
# Sample many x_T values and plot the pixel value distribution
N = 500
x0_batch = torch.stack([mnist[i][0] for i in range(N)]).to(device)  # (N, 1, 28, 28)
t_final  = torch.full((N,), 999, device=device, dtype=torch.long)
xt_final, _ = linear_sched.add_noise(x0_batch, t_final)

pixel_values = xt_final.cpu().numpy().flatten()

x_plot = np.linspace(-4, 4, 300)
gaussian_pdf = np.exp(-0.5 * x_plot**2) / np.sqrt(2 * np.pi)

plt.figure(figsize=(7, 4))
plt.hist(pixel_values, bins=100, density=True, alpha=0.7, label="x_T pixel values")
plt.plot(x_plot, gaussian_pdf, "r-", linewidth=2, label="N(0,1)")
plt.xlabel("Pixel value"); plt.ylabel("Density")
plt.title("Distribution of x_T (t=999) — should match N(0,1)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Mean of x_T: {pixel_values.mean():.4f}  (expect ≈ 0)")
print(f"Std  of x_T: {pixel_values.std():.4f}  (expect ≈ 1)")

## Section 8 — Final Reflection

**What clicked for me this week:**

This was genuinely the hardest week conceptually, mostly because getting the notation straight took longer than expected. I spent a while confusing α_t with ᾱ_t — they look similar but they're very different things. α_t is per-step, ᾱ_t is cumulative, and almost everything you actually use in practice involves ᾱ_t.

The moment that really clicked was working through the algebra to see why the closed-form works. It's not magic — it's just the property that if $X \sim \mathcal{N}(\mu_1, \sigma_1^2)$ and $Y \sim \mathcal{N}(\mu_2, \sigma_2^2)$ independently, then $aX + bY \sim \mathcal{N}(a\mu_1 + b\mu_2, a^2\sigma_1^2 + b^2\sigma_2^2)$. Apply that recursively across all T steps and the β's multiply together into ᾱ_t.

The SNR plot was also really illuminating. You can visually see why the linear schedule is suboptimal — it blows through most of the "interesting" noise levels very quickly and then spends the majority of timesteps in the near-pure-noise regime where the model can't learn anything useful.

Going into Week 4, I now have a `NoiseScheduler` that can produce `(x_t, t, noise)` tuples efficiently during training — no iterating required. The UNet just needs to learn to predict `noise` from `(x_t, t)`.